# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [100]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [101]:
# TODO: Import the necessary libs
# For example: 
# import os

# from lib.agents import Agent
# from lib.llm import LLM
# from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
# from lib.tooling import tool

import os

from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [102]:
# TODO: Load environment variables
# load_dotenv()

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
# TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [103]:
print("OPENAI_API_KEY loaded:", OPENAI_API_KEY is not None)
print("TAVILY_API_KEY loaded:", TAVILY_API_KEY is not None)

OPENAI_API_KEY loaded: True
TAVILY_API_KEY loaded: True


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [104]:
import logging

import chromadb
from chromadb.config import Settings

# Suppress non-critical ChromaDB telemetry log messages.
logging.getLogger(
    "chromadb.telemetry.product.posthog"
).setLevel(logging.CRITICAL)

# Disable anonymous telemetry for the ChromaDB client.
chroma_settings = Settings(
    anonymized_telemetry=False
)


In [105]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

import chromadb

chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB.

    Args:
        query: A question about the game industry.

    Returns:
        A list of relevant game records.
    """
    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    games = []

    for metadata in results["metadatas"][0]:
        games.append({
            "Platform": metadata.get("Platform"),
            "Name": metadata.get("Name"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description")
        })

    return games

In [106]:
retrieved_docs = retrieve_game(
    "When was FIFA 21 released and on which platform?"
)

print("Retrieved documents:", len(retrieved_docs))
print(
    "Retrieved games:",
    [document["Name"] for document in retrieved_docs]
)

Retrieved documents: 5
Retrieved games: ['Halo Infinite', 'Gran Turismo', 'Minecraft', 'Mario Kart 8 Deluxe', 'Gran Turismo 5']


#### Evaluate Retrieval Tool

In [107]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

from pydantic import BaseModel, Field

from lib.parsers import PydanticOutputParser


class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question"
    )
    description: str = Field(
        description="Detailed explanation of the retrieval evaluation"
    )


evaluation_llm = LLM(model="gpt-4o-mini")


@tool
def evaluate_retrieval(
    question: str,
    retrieved_docs: list
) -> dict:
    """
    Analyze whether retrieved documents can answer the user's question.

    Args:
        question: The original question from the user.
        retrieved_docs: Documents retrieved from the vector database.

    Returns:
        A structured retrieval evaluation.
    """
    prompt = f"""
Your task is to evaluate whether the retrieved documents are sufficient
to answer the user's question.

User question:
{question}

Retrieved documents:
{retrieved_docs}

Mark useful as true only when the documents contain enough relevant
information to answer the question accurately.

Provide a detailed description explaining why the documents should be
accepted or rejected.
"""

    response = evaluation_llm.invoke(
        input=prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)

    return evaluation.model_dump()

In [108]:
retrieved_docs = retrieve_game(
    "When was FIFA 21 released and on which platform?"
)

evaluation_result = evaluate_retrieval(
    question="When was FIFA 21 released and on which platform?",
    retrieved_docs=retrieved_docs
)

evaluation_result

{'useful': False,
 'description': "The retrieved documents do not contain any relevant information regarding the release date or platforms for FIFA 21. Instead, they provide details about other video games, including their release years and platforms, but none of these games are FIFA 21. Therefore, the documents are insufficient to answer the user's question."}

#### Game Web Search Tool

In [109]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

from tavily import TavilyClient


tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)


@tool
def game_web_search(question: str) -> list:
    """
    Search the web for current information about the game industry.

    Args:
        question: A question about the game industry.

    Returns:
        A list of relevant web search results.
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )

    search_results = []

    for result in response.get("results", []):
        search_results.append({
            "title": result.get("title"),
            "content": result.get("content"),
            "url": result.get("url")
        })

    return search_results

In [110]:
game_web_search(
    "When was FIFA 21 released and on which platforms?"
)

[{'title': 'FIFA 21',
  'content': 'FIFA 21 is an association football simulation video game published by Electronic Arts as part of the FIFA series "FIFA (video game series)"). It is the 28th installment in the FIFA series, and was released on 9 October 2020 for Microsoft Windows, Nintendo Switch, PlayStation 4, and Xbox One. Enhanced versions for the PlayStation 5 and Xbox Series X and Series S were released on 3 December 2020, in addition to a version for Stadia in March 2021. The online servers for the game for Stadia were shut down on 18 January 2023, whilst the online servers for all other platforms were shut down on 6 November 2023.\n\n## Features\n\n[edit]\n\n### Ultimate Team\n\n[edit] [...] ## Release\n\n[edit]\n\nFIFA 21 was released worldwide on 9 October 2020 for Microsoft Windows, PlayStation 4, Xbox One and Nintendo Switch. As with previous installments, the Switch version is a "Legacy Edition", that contains updated kits, rosters, and minor updates, but does not include

In [111]:
retrieved_docs = retrieve_game(
    "When was FIFA 21 released and on which platform?"
)

retrieved_docs

[{'Platform': 'Xbox Series X|S',
  'Name': 'Halo Infinite',
  'YearOfRelease': 2021,
  'Description': "The latest installment in the Halo franchise, featuring Master Chief's return in a new open-world setting."},
 {'Platform': 'PlayStation 1',
  'Name': 'Gran Turismo',
  'YearOfRelease': 1997,
  'Description': 'A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.'},
 {'Platform': 'Xbox One',
  'Name': 'Minecraft',
  'YearOfRelease': 2014,
  'Description': 'A sandbox game that allows players to build and explore infinite worlds, fostering creativity and adventure.'},
 {'Platform': 'Nintendo Switch',
  'Name': 'Mario Kart 8 Deluxe',
  'YearOfRelease': 2017,
  'Description': 'An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.'},
 {'Platform': 'PlayStation 3',
  'Name': 'Gran Turismo 5',
  'YearOfRelease': 2010,
  'Description': 'A comprehensive racing simulator featuring a vast se

In [112]:
web_results = game_web_search(
    "When was FIFA 21 released and on which platforms?"
)

print("Number of web results:", len(web_results))

for index, result in enumerate(web_results, start=1):
    print(f"\nResult {index}")
    print("Title:", result.get("title"))
    print("Content:", result.get("content"))
    print("URL:", result.get("url"))

Number of web results: 5

Result 1
Title: FIFA 21
Content: FIFA 21 is an association football simulation video game published by Electronic Arts as part of the FIFA series "FIFA (video game series)"). It is the 28th installment in the FIFA series, and was released on 9 October 2020 for Microsoft Windows, Nintendo Switch, PlayStation 4, and Xbox One. Enhanced versions for the PlayStation 5 and Xbox Series X and Series S were released on 3 December 2020, in addition to a version for Stadia in March 2021. The online servers for the game for Stadia were shut down on 18 January 2023, whilst the online servers for all other platforms were shut down on 6 November 2023.

## Features

[edit]

### Ultimate Team

[edit] [...] ## Release

[edit]

FIFA 21 was released worldwide on 9 October 2020 for Microsoft Windows, PlayStation 4, Xbox One and Nintendo Switch. As with previous installments, the Switch version is a "Legacy Edition", that contains updated kits, rosters, and minor updates, but does 

### Agent

In [113]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

agent_instructions = """
You are UdaPlay, an AI research agent for the video game industry.

For every user question, follow this workflow:

1. Always call retrieve_game first to search the internal vector database.
2. Pass the original user question and the retrieved documents to
   evaluate_retrieval.
3. Inspect the evaluation result:
   - If useful is true, answer using the retrieved documents.
   - If useful is false, call game_web_search with the original question.
4. Use only information returned by the tools.
5. Do not invent missing facts.
6. Clearly state when the internal knowledge was insufficient and web
   search was required.
7. Provide a clear and concise final answer.
8. Include the relevant source URL when web search results are used.
9. Preserve the conversation context for follow-up questions in the
   same session.
"""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=agent_instructions,
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ],
    temperature=0.0
)

In [114]:
print("Agent type:", type(agent))
print("Registered tools:", [tool.name for tool in agent.tools])

Agent type: <class 'lib.agents.Agent'>
Registered tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']


In [115]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

first_run = agent.invoke(
    query="When Pokémon Gold and Silver was released?",
    session_id="udaplay_demo"
)

first_run

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


Run('9f8303ae-159f-46c6-89d2-4a2ab310f3f3')

In [116]:
print(type(first_run))
print([name for name in dir(first_run) if not name.startswith("_")])

<class 'lib.state_machine.Run'>
['add_snapshot', 'complete', 'create', 'end_timestamp', 'get_final_state', 'metadata', 'run_id', 'snapshots', 'start_timestamp']


In [117]:
final_state = first_run.get_final_state()

print("Run complete:", first_run.complete)
print("Final state fields:", list(final_state.keys()))
print("Number of messages:", len(final_state.get("messages", [])))

for index, message in enumerate(final_state.get("messages", []), start=1):
    print(f"\nMessage {index}")
    print("Type:", type(message).__name__)
    print("Content:", message.content)

    tool_calls = getattr(message, "tool_calls", None)

    if tool_calls:
        print(
            "Tool calls:",
            [call.function.name for call in tool_calls]
        )

print("\nFinal answer:")
print(final_state.get("messages", [])[-1].content)

Run complete: <bound method Run.complete of Run('9f8303ae-159f-46c6-89d2-4a2ab310f3f3')>
Final state fields: ['user_query', 'instructions', 'messages', 'current_tool_calls', 'session_id', 'total_tokens']
Number of messages: 7

Message 1
Type: SystemMessage
Content: 
You are UdaPlay, an AI research agent for the video game industry.

For every user question, follow this workflow:

1. Always call retrieve_game first to search the internal vector database.
2. Pass the original user question and the retrieved documents to
   evaluate_retrieval.
3. Inspect the evaluation result:
   - If useful is true, answer using the retrieved documents.
   - If useful is false, call game_web_search with the original question.
4. Use only information returned by the tools.
5. Do not invent missing facts.
6. Clearly state when the internal knowledge was insufficient and web
   search was required.
7. Provide a clear and concise final answer.
8. Include the relevant source URL when web search results are us

In [118]:
second_run = agent.invoke(
    query="Which one was the first 3D platformer Mario game?",
    session_id="udaplay_demo_2"
)

second_final_state = second_run.get_final_state()

second_tool_names = []

for message in second_final_state["messages"]:
    if getattr(message, "tool_calls", None):
        second_tool_names.extend(
            call.function.name
            for call in message.tool_calls
        )

print("Run complete:", second_run.complete())
print("Tools used:", second_tool_names)
print("Final answer:", second_final_state["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Run complete: None
Tools used: ['retrieve_game', 'evaluate_retrieval']
Final answer: The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game was groundbreaking and set new standards for the genre, featuring Mario's quest to rescue Princess Peach.


In [119]:
third_run = agent.invoke(
    query="Was Mortal Kombat X released for PlayStation 5?",
    session_id="udaplay_demo_3"
)

third_final_state = third_run.get_final_state()

third_tool_names = []

for message in third_final_state["messages"]:
    if getattr(message, "tool_calls", None):
        third_tool_names.extend(
            call.function.name
            for call in message.tool_calls
        )

print("Tools used:", third_tool_names)
print("Final answer:", third_final_state["messages"][-1].content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Tools used: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
Final answer: Mortal Kombat X was not originally released for PlayStation 5. It was developed for PlayStation 4, Xbox One, and Microsoft Windows, and its release date was April 14, 2015. However, it is playable on PlayStation 5 through backward compatibility, although some features available on the PS4 version may be absent. 

For more details, you can check the PlayStation Store page [here](https://store.playstation.com/en-us/product/UP1018-CUSA00967_00-MORTALKOMBATX000).


In [121]:
from IPython.display import clear_output


clear_output(wait=True)


def display_run_summary(title: str, run) -> None:
    final_state = run.get_final_state()
    tools_used = []

    for message in final_state["messages"]:
        if getattr(message, "tool_calls", None):
            tools_used.extend(
                call.function.name
                for call in message.tool_calls
            )

    print("=" * 70)
    print(title)
    print("=" * 70)
    print("Question:")
    print(final_state["user_query"])
    print()
    print("Tools used:")
    print(" -> ".join(tools_used) if tools_used else "No tools used")
    print()
    print("Final answer:")
    print(final_state["messages"][-1].content)
    print()


display_run_summary(
    "Example Query 1",
    first_run
)

display_run_summary(
    "Example Query 2",
    second_run
)

display_run_summary(
    "Example Query 3",
    third_run
)


Example Query 1
Question:
When Pokémon Gold and Silver was released?

Tools used:
retrieve_game -> evaluate_retrieval

Final answer:
Pokémon Gold and Silver was released in 1999 for the Game Boy Color.

Example Query 2
Question:
Which one was the first 3D platformer Mario game?

Tools used:
retrieve_game -> evaluate_retrieval

Final answer:
The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996 for the Nintendo 64. This game was groundbreaking and set new standards for the genre, featuring Mario's quest to rescue Princess Peach.

Example Query 3
Question:
Was Mortal Kombat X released for PlayStation 5?

Tools used:
retrieve_game -> evaluate_retrieval -> game_web_search

Final answer:
Mortal Kombat X was not originally released for PlayStation 5. It was developed for PlayStation 4, Xbox One, and Microsoft Windows, and its release date was April 14, 2015. However, it is playable on PlayStation 5 through backward compatibility, although some features availabl

### (Optional) Advanced

In [120]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes